# Kopi IoT Web3 — Deep Learning Forecasting (Keras) — Multi-step + Multi-seed

Mengajarkan **ANN → DNN → LSTM** + membandingkan dengan **Naive, Linear, Prophet** untuk
**3 variabel** (suhu, udara, tanah).

- **Evaluasi SETARA (multi-step recursive):** semua model memprediksi K hari ke depan tanpa melihat aktual.
- **Multi-seed:** tiap model DL dilatih beberapa kali (seed berbeda) → dilaporkan **rata-rata ± std**.
  Sebab pada data sedikit, hasil DL **bervariasi**; rata-rata banyak run lebih jujur daripada satu run.

> Kurva Loss/MAE per epoch OTENTIK (neural network). Catat di laporan: metode evaluasi, ukuran data, jumlah seed.

Jalankan: Runtime -> Run all. (±3–5 menit: 3 seed × 3 model × 3 variabel.)

In [ ]:
!pip -q install prophet

## 1. Ambil & bersihkan data

In [ ]:
import requests, pandas as pd, numpy as np

CHANNEL_ID = 1848413
url = f'https://api.thingspeak.com/channels/{CHANNEL_ID}/feeds.json'
feeds = requests.get(url, params={'results':8000,'timezone':'Asia/Jakarta'}, timeout=30).json()['feeds']

df = pd.DataFrame(feeds)
df['created_at'] = pd.to_datetime(df['created_at'])
if df['created_at'].dt.tz is not None: df['created_at'] = df['created_at'].dt.tz_localize(None)
for col,name in [('field1','suhu'),('field2','udara'),('field3','tanah')]:
    df[name] = pd.to_numeric(df[col], errors='coerce')
df = df[['created_at','suhu','udara','tanah']].set_index('created_at')

RANGES = {'suhu':(10,45),'udara':(0,100),'tanah':(0,100)}
for v,(lo,hi) in RANGES.items(): df[v] = df[v].where((df[v]>=lo)&(df[v]<=hi))
for v in RANGES:
    s=df[v]; med=s.median(); mad=(s-med).abs().median()
    if mad and mad>0: df[v]=s.where((s-med).abs()<=3.5*1.4826*mad)

harian = df.resample('D').mean().interpolate(method='time', limit=3, limit_area='inside').dropna(how='all')
print('Jumlah hari:', len(harian)); harian.tail()

## 2. Metrik & arsitektur model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.linear_model import LinearRegression
from prophet import Prophet
import logging
logging.getLogger('prophet').setLevel(logging.WARNING); logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

def mae(a,p):  a,p=np.asarray(a,float),np.asarray(p,float); return float(np.mean(np.abs(a-p)))
def rmse(a,p): a,p=np.asarray(a,float),np.asarray(p,float); return float(np.sqrt(np.mean((a-p)**2)))
def mape(a,p):
    a,p=np.asarray(a,float),np.asarray(p,float); m=np.abs(a)>1e-6
    return float(np.mean(np.abs((a[m]-p[m])/a[m]))*100) if m.sum() else float('nan')
def akurasi(a,p): return float(max(0.0, 100-mape(a,p)))

def build_mlp(W):  m=models.Sequential([layers.Input((W,)), layers.Dense(16,activation='relu'), layers.Dense(1)]); m.compile(optimizer='adam', loss='mse', metrics=['mae']); return m
def build_dnn(W):  m=models.Sequential([layers.Input((W,)), layers.Dense(32,activation='relu'), layers.Dropout(0.2), layers.Dense(16,activation='relu'), layers.Dense(1)]); m.compile(optimizer='adam', loss='mse', metrics=['mae']); return m
def build_lstm(W): m=models.Sequential([layers.Input((W,1)), layers.LSTM(16), layers.Dense(1)]); m.compile(optimizer='adam', loss='mse', metrics=['mae']); return m

## 3. Fungsi analisis (multi-step, multi-seed)

In [ ]:
SEEDS = (42, 7, 123)   # jumlah & nilai seed bisa diubah

def analisis(VAR, W=7, K=10, epochs=300, seeds=SEEDS):
    s = harian[VAR].dropna(); dates = list(s.index); vals = s.values.astype('float32')
    train_vals, test_actual, test_dates = vals[:-K], vals[-K:], dates[-K:]

    # windowing dari TRAIN
    X,y = [],[]
    for i in range(len(train_vals)-W): X.append(train_vals[i:i+W]); y.append(train_vals[i+W])
    X,y = np.array(X), np.array(y)
    mn,mx = float(X.min()), float(X.max())
    norm   = lambda a:(a-mn)/(mx-mn+1e-9)
    denorm = lambda a:a*(mx-mn+1e-9)+mn
    lw = train_vals[-W:]

    def recursive(step):
        win = lw.astype('float32').copy(); out=[]
        for _ in range(K): yh=step(win); out.append(yh); win=np.append(win[1:], yh)
        return np.array(out)

    # ── Baseline deterministik (multi-step) ──
    p_naive = np.repeat(float(train_vals[-1]), K)
    lr = LinearRegression().fit(X, y)
    p_lin = recursive(lambda w: float(lr.predict(w.reshape(1,-1))[0]))
    dfp = harian[[VAR]].dropna().reset_index(); dfp.columns=['ds','y']
    if dfp['ds'].dt.tz is not None: dfp['ds']=dfp['ds'].dt.tz_localize(None)
    mP = Prophet(weekly_seasonality=False, daily_seasonality=False, yearly_seasonality=False, changepoint_prior_scale=0.01)
    mP.fit(dfp[dfp['ds'] < test_dates[0]])
    p_prophet = mP.predict(pd.DataFrame({'ds':test_dates}))['yhat'].values

    # ── Deep learning: ulangi tiap seed ──
    arch = [('MLP (ANN)', build_mlp, False), ('DNN', build_dnn, False), ('LSTM', build_lstm, True)]
    skor = {n: {'mae':[], 'rmse':[], 'akur':[]} for n,_,_ in arch}
    hist_keep, pred_keep = {}, {}
    for sd in seeds:
        tf.keras.utils.set_random_seed(int(sd))
        for nama, build, reshape in arch:
            m = build(W)
            Xfit = norm(X).reshape(-1,W,1) if reshape else norm(X)
            h = m.fit(Xfit, norm(y), validation_split=0.2, epochs=epochs, batch_size=8, verbose=0,
                      callbacks=[callbacks.EarlyStopping(patience=40, restore_best_weights=True)])
            if reshape: step = lambda w, mm=m: float(denorm(mm.predict(norm(w).reshape(1,W,1), verbose=0)[0,0]))
            else:       step = lambda w, mm=m: float(denorm(mm.predict(norm(w).reshape(1,-1),  verbose=0)[0,0]))
            p = recursive(step)
            skor[nama]['mae'].append(mae(test_actual,p)); skor[nama]['rmse'].append(rmse(test_actual,p)); skor[nama]['akur'].append(akurasi(test_actual,p))
            hist_keep[nama] = h; pred_keep[nama] = p   # simpan run terakhir untuk grafik

    rows = []
    for nama, p in [('Naive',p_naive), ('Linear',p_lin), ('Prophet',p_prophet)]:
        rows.append({'model':nama, 'MAE':round(mae(test_actual,p),3), 'MAE_std':0.0, 'RMSE':round(rmse(test_actual,p),3), 'Akurasi(%)':round(akurasi(test_actual,p),1)})
    for nama, sc in skor.items():
        rows.append({'model':nama, 'MAE':round(np.mean(sc['mae']),3), 'MAE_std':round(np.std(sc['mae']),3), 'RMSE':round(np.mean(sc['rmse']),3), 'Akurasi(%)':round(np.mean(sc['akur']),1)})
    tabel = pd.DataFrame(rows).sort_values('MAE').reset_index(drop=True)

    preds = {'Naive':p_naive, 'Linear':p_lin, 'Prophet':p_prophet, **pred_keep}
    return {'VAR':VAR, 'tabel':tabel, 'hist':hist_keep, 'preds':preds, 'actual':test_actual, 'dates':test_dates, 'nseed':len(seeds)}

## 4. Fungsi grafik

In [ ]:
import matplotlib.pyplot as plt

def plot_loss(res):
    fig,ax = plt.subplots(3,2,figsize=(12,11))
    for row,nama in enumerate(['MLP (ANN)','DNN','LSTM']):
        h=res['hist'][nama]
        ax[row,0].plot(h.history['loss'],label='train'); ax[row,0].plot(h.history['val_loss'],label='val')
        ax[row,0].set_title(nama+' - Loss (MSE) - '+res['VAR']); ax[row,0].set_xlabel('epoch'); ax[row,0].grid(alpha=.3); ax[row,0].legend()
        ax[row,1].plot(h.history['mae'],label='train'); ax[row,1].plot(h.history['val_mae'],label='val')
        ax[row,1].set_title(nama+' - MAE'); ax[row,1].set_xlabel('epoch'); ax[row,1].grid(alpha=.3); ax[row,1].legend()
    plt.tight_layout(); plt.show()

def plot_compare(res):
    tt=res['tabel'].set_index('model'); idx=range(len(tt))
    fig,(a1,a2)=plt.subplots(1,2,figsize=(13,4))
    a1.bar(idx, tt['MAE'], yerr=tt['MAE_std'], color='#ef4444', capsize=4)
    a1.set_xticks(list(idx)); a1.set_xticklabels(tt.index, rotation=30, ha='right'); a1.set_title('MAE +/- std - '+res['VAR']); a1.grid(alpha=.3,axis='y')
    a2.bar(idx, tt['Akurasi(%)'], color='#16a34a'); a2.set_xticks(list(idx)); a2.set_xticklabels(tt.index, rotation=30, ha='right'); a2.set_ylim(0,100); a2.set_title('Akurasi - '+res['VAR']); a2.grid(alpha=.3,axis='y')
    plt.tight_layout(); plt.show()

def plot_actual(res):
    cmap={'Naive':'#9ca3af','Linear':'#a78bfa','Prophet':'#16a34a','MLP (ANN)':'#ef4444','DNN':'#f59e0b','LSTM':'#3b82f6'}
    plt.figure(figsize=(11,4)); plt.plot(res['dates'],res['actual'],'o-',color='#111',label='Aktual',linewidth=2)
    for nama,p in res['preds'].items(): plt.plot(res['dates'],p,'--',color=cmap.get(nama),label=nama,alpha=.85)
    plt.title('Aktual vs Prediksi (multi-step) - '+res['VAR']); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 5. Variabel: SUHU (dengan kurva loss)

In [ ]:
res_suhu = analisis('suhu')
print('=== SUHU (DL: rata-rata', res_suhu['nseed'], 'seed) ==='); print(res_suhu['tabel'].to_string(index=False))
plot_loss(res_suhu)

In [ ]:
plot_compare(res_suhu); plot_actual(res_suhu)

## 6. Variabel: UDARA

In [ ]:
res_udara = analisis('udara')
print('=== UDARA ==='); print(res_udara['tabel'].to_string(index=False))
plot_compare(res_udara); plot_actual(res_udara)

## 7. Variabel: TANAH

In [ ]:
res_tanah = analisis('tanah')
print('=== TANAH ==='); print(res_tanah['tabel'].to_string(index=False))
plot_compare(res_tanah); plot_actual(res_tanah)

## 8. Ringkasan gabungan (semua variabel)

In [ ]:
gab = pd.concat([r['tabel'].assign(variabel=r['VAR']) for r in [res_suhu,res_udara,res_tanah]])
print('=== MAE per model x variabel ==='); print(gab.pivot(index='model',columns='variabel',values='MAE').round(3).to_string())
print(); print('=== MAE_std (kestabilan DL) ==='); print(gab.pivot(index='model',columns='variabel',values='MAE_std').round(3).to_string())
print(); print('=== Akurasi(%) per model x variabel ==='); print(gab.pivot(index='model',columns='variabel',values='Akurasi(%)').round(1).to_string())
gab.pivot(index='model',columns='variabel',values='MAE').plot(kind='bar',figsize=(10,4)); plt.title('MAE per model (multi-step, rata-rata seed)'); plt.ylabel('MAE'); plt.grid(alpha=.3,axis='y'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 9. Kesimpulan

- Perbandingan **setara** (multi-step) & **kredibel** (rata-rata ± std banyak seed).
- **MAE_std besar = model tidak stabil** pada data sedikit (lihat tabel std) — argumen kuat bahwa DL butuh lebih banyak data.
- Pemenang **berbeda tiap variabel** → membenarkan pendekatan *best-per-variable*.
- Tangga ajar: Naive -> Linear -> Prophet -> ANN -> DNN -> LSTM.
- Laporan: sajikan MAE/RMSE/Akurasi (rata-rata ± std), metode evaluasi (multi-step), ukuran data, & jumlah seed.